# Modular TensorLLM / TT / LoRA / quantization experiments

In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0
%config InlineBackend.close_figures = False

In [ ]:
import tqdm as _tqdm_module
from tqdm.auto import tqdm as _auto_tqdm

_tqdm_module.tqdm = _auto_tqdm

In [ ]:
import os
os.environ.pop("TORCH_LOGS", None)

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'src').exists() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)
print('Has src:', (REPO_ROOT / 'src').exists())

In [ ]:
import gc
import json
import math
from dataclasses import asdict

import torch
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from src.modular_compression_experiment import (
    MODEL_ARCH_REGISTRY,
    ModelArchitectureSpec,
    ModulePlan,
    RecipeSpec,
    RunnerConfig,
    add_model_architecture,
    clean_mem,
    completed_keys_from_results,
    get_num_layers_from_config,
    get_model_architecture,
    load_model_and_tokenizer,
    make_target_specs,
    middle_expanding_layer_sets,
    run_experiment_grid,
    save_results,
    load_results_checkpoint,
)

from src.head_decomp_methods import *

pd.set_option('display.max_colwidth', 260)
torch.set_grad_enabled(True)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

print('Registered model architectures:', sorted(MODEL_ARCH_REGISTRY))

## Configuration

In [ ]:
START = 1
END = 32

In [ ]:
OUTPUT_DIR = REPO_ROOT / 'results_per_layer_tt_ffn'
OUTPUT_JSON = OUTPUT_DIR / 'llama_tt_tensorllm_results.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_GPTJ = False
RUN_LLAMA = False
RUN_LLAMA2 = True

GPTJ_MODEL_NAME = 'EleutherAI/gpt-j-6B'
LLAMA_8B_MODEL_NAME = 'meta-llama/Llama-2-7b-hf'

MODEL_DTYPE = torch.float16
DEVICE_MAP = {"": 0}
MODEL_STORAGE_BITS = 16
LORA_STORAGE_BITS = 16
SCALE_STORAGE_BITS = 16

# Decomposition / quantization.
RANKS = [64]
BITWIDTH = 4
ORDER = 12
TOKEN_CHUNK_SIZE = 16
DECOMPOSE_DTYPE = torch.float64
DECOMPOSE_DEVICE = 'cuda' # 'cpu'
DENSE_SPARSE_OUTLIER_FRACTION = 5e-7

TENSORLLM_STACK_RANK = 2
TENSORLLM_HEAD_DIM_RANK = 4
TENSORLLM_TUCKER_TYPE = 'partial_tucker_v5'
TENSORLLM_ALLOW_GQA_PER_PROJECTION_FALLBACK = True

# LoRA defaults.
LORA_R = 0
LORA_ALPHA = 0
LORA_DROPOUT = 0.0
LEARNING_RATE = 0
WEIGHT_DECAY = 0.0
MAX_STEPS = 0
GRAD_ACCUM_STEPS = 8
TRAIN_BATCH_SIZE = 0
TRAIN_SEQ_LEN = 0
NUM_TRAIN_SEQUENCES = 0

# Benchmarks.
BENCHMARK_WITH_DENSE_RECONSTRUCTION = True
RUN_PPL = True
PPL_DATASETS = ['wikitext2', 'c4']
PPL_SEQLEN = 2048
SEQ_LEN_BY_ARCH = {
    "gptj": {
        "ppl_seqlen": 2048,
        "lora_train_seq_len": 256,
        "geometry_seq_len": 512,
    },
    "llama": {
        "ppl_seqlen": 4096,
        "lora_train_seq_len": 512,
        "geometry_seq_len": 1024,
    },
    "llama2": {
        "ppl_seqlen": 4096,
        "lora_train_seq_len": 512,
        "geometry_seq_len": 1024,
    },
}

RUN_LM_EVAL = False
LM_EVAL_TASKS = ['arc_challenge', 'winogrande', 'piqa', 'hellaswag', 'openbookqa']
LM_EVAL_NUM_FEWSHOT = 0
LM_EVAL_BATCH_SIZE = 1
LM_EVAL_LIMIT = None

RUN_HOTPOT_EVAL = False
HOTPOT_MAX_EXAMPLES = None
HOTPOT_BATCH_SIZE = 16
HOTPOT_MAX_NEW_TOKENS = 15
HOTPOT_BEAM = 1

RUN_ACTIVATION_GEOMETRY = True
GEOMETRY_NUM_SAMPLES = 100
GEOMETRY_SEQ_LEN = 512
GEOMETRY_SEED = 42

RUN_GENERATION_EXAMPLES = True
GENERATION_PROMPTS = [
    'The theory of tensor train decomposition for neural networks suggests that',
    'Apple Inc. is a worldwide tech company because',
    'Summer is hot. Winter is',
    'Sylvester Stallone is best known for',
    'Sharpness-aware minimization is',
]
GENERATION_MAX_NEW_TOKENS = 80

## Model runner configs

In [ ]:
def make_cfg(model_arch_key: str, model_name: str) -> RunnerConfig:
    seq_cfg = SEQ_LEN_BY_ARCH.get(model_arch_key, {})

    return RunnerConfig(
        model_arch_key=model_arch_key,
        model_name=model_name,
        model_dtype=MODEL_DTYPE,
        model_storage_bits=MODEL_STORAGE_BITS,
        lora_storage_bits=LORA_STORAGE_BITS,
        scale_storage_bits=SCALE_STORAGE_BITS,
        device_map=DEVICE_MAP,
        output_dir=OUTPUT_DIR,
        decompose_dtype=DECOMPOSE_DTYPE,
        decompose_device=DECOMPOSE_DEVICE,
        tt_order=ORDER,
        token_chunk_size=TOKEN_CHUNK_SIZE,
        dense_sparse_outlier_fraction=DENSE_SPARSE_OUTLIER_FRACTION,
        tensorllm_stack_rank=TENSORLLM_STACK_RANK,
        tensorllm_head_dim_rank=TENSORLLM_HEAD_DIM_RANK,
        tensorllm_tucker_type=TENSORLLM_TUCKER_TYPE,
        tensorllm_allow_gqa_per_projection_fallback=TENSORLLM_ALLOW_GQA_PER_PROJECTION_FALLBACK,

        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_lr=LEARNING_RATE,
        lora_weight_decay=WEIGHT_DECAY,
        lora_max_steps=MAX_STEPS,
        lora_grad_accum_steps=GRAD_ACCUM_STEPS,
        lora_train_batch_size=TRAIN_BATCH_SIZE,
        lora_train_seq_len=seq_cfg.get("lora_train_seq_len", TRAIN_SEQ_LEN),
        lora_num_train_sequences=NUM_TRAIN_SEQUENCES,

        run_ppl=RUN_PPL,
        ppl_datasets=PPL_DATASETS,
        ppl_seqlen=seq_cfg.get("ppl_seqlen", PPL_SEQLEN),

        run_lm_eval=RUN_LM_EVAL,
        lm_eval_tasks=LM_EVAL_TASKS,
        lm_eval_num_fewshot=LM_EVAL_NUM_FEWSHOT,
        lm_eval_batch_size=LM_EVAL_BATCH_SIZE,
        lm_eval_limit=LM_EVAL_LIMIT,

        run_hotpot=RUN_HOTPOT_EVAL,
        hotpot_max_examples=HOTPOT_MAX_EXAMPLES,
        hotpot_batch_size=HOTPOT_BATCH_SIZE,
        hotpot_max_new_tokens=HOTPOT_MAX_NEW_TOKENS,
        hotpot_beam=HOTPOT_BEAM,

        run_activation_geometry=RUN_ACTIVATION_GEOMETRY,
        geometry_num_samples=GEOMETRY_NUM_SAMPLES,
        geometry_seq_len=seq_cfg.get("geometry_seq_len", GEOMETRY_SEQ_LEN),
        geometry_seed=GEOMETRY_SEED,

        run_generation_examples=RUN_GENERATION_EXAMPLES,
        generation_prompts=GENERATION_PROMPTS,
        generation_max_new_tokens=GENERATION_MAX_NEW_TOKENS,

        use_torch_compile_for_benchmarks=True,
        enable_tf32_for_benchmarks=True,
        torch_compile_mode="reduce-overhead",
        torch_compile_fullgraph=False,
        torch_compile_dynamic=None,
    )

RUNNER_CONFIGS = []
if RUN_GPTJ:
    RUNNER_CONFIGS.append(make_cfg('gptj', GPTJ_MODEL_NAME))
if RUN_LLAMA:
    RUNNER_CONFIGS.append(make_cfg('llama', LLAMA_8B_MODEL_NAME))
if RUN_LLAMA2:
    RUNNER_CONFIGS.append(make_cfg('llama2', LLAMA_8B_MODEL_NAME))

pd.DataFrame([
    {
        "model_arch_key": c.model_arch_key,
        "model_name": c.model_name,
        "dtype": str(c.model_dtype),
        "ppl_datasets": c.ppl_datasets,
        "ppl_seqlen": c.ppl_seqlen,
        "lora_train_seq_len": c.lora_train_seq_len,
        "geometry_seq_len": c.geometry_seq_len,
        "run_lm_eval": c.run_lm_eval,
        "lm_eval_tasks": c.lm_eval_tasks,
    }
    for c in RUNNER_CONFIGS
])

## Target layer sets

In [ ]:
def itterate_over_layers(n_layers, 
                        start=None,
                        end=None):
    out = []
    
    if start is not None and end is not None:
        for i in range(start, end):
            conf = {'label': f"llama_all_layer_{i}",
                'layer_indices': [i],
                'groups': ['mha', 'mlp'],
                'known_superweights': {}}
            out.append(conf)
        
        return out

    for i in range(n_layers):
        conf = {'label': f"llama_all_layer_{i}",
                'layer_indices': [i],
                'groups': ['mha', 'mlp'],
                'known_superweights': {}}
        out.append(conf)

    return out

In [ ]:
TARGET_SETS_BY_MODEL = {}
for cfg in RUNNER_CONFIGS:
    '''
    n_layers = get_num_layers_from_config(cfg.model_name)
    TARGET_SETS_BY_MODEL[cfg.model_name] = middle_expanding_layer_sets(
        n_layers,
        label_prefix=f'{cfg.model_arch_key}_middle',
        max_fraction=0.25,
    )
    print(cfg.model_name, 'num_layers=', n_layers, 'num_target_sets=', len(TARGET_SETS_BY_MODEL[cfg.model_name]))
    display(pd.DataFrame(TARGET_SETS_BY_MODEL[cfg.model_name]))
    '''
    n_layers = get_num_layers_from_config(cfg.model_name)

    TARGET_SETS_BY_MODEL[cfg.model_name] = itterate_over_layers(
        n_layers,
        start=START,
        end=END
    )
    print(cfg.model_name, 'num_layers=', n_layers, 'num_target_sets=', len(TARGET_SETS_BY_MODEL[cfg.model_name]))
    display(pd.DataFrame(TARGET_SETS_BY_MODEL[cfg.model_name]))

## Recipes

In [ ]:
def make_16x_recipes(
    *,
    quant_bits: int = 4,
    include_lora: bool = True,
    benchmark_with_dense_reconstruction: bool = True,
) -> list[RecipeSpec]:
    """
      1. Tucker on MHA
      2. Tucker on MHA + TT on FFN/MLP
      3. TT on FFN/MLP
      4. TT on all selected modules

    Variants:
      - quantized / unquantized
      - with LoRA / without LoRA
    """
    base = [
        ("tucker_mha", [
            ModulePlan("mha", "tensorllm_tucker_mha", quant_method="none"),
        ]),
        ("tucker_mha__tt_mlp", [
            ModulePlan("mha", "tensorllm_tucker_mha", quant_method="none"),
            ModulePlan("mlp", "tt", quant_method="none"),
        ]),
        ("tt_mlp", [
            ModulePlan("mlp", "tt", quant_method="none"),
        ]),
        ("tt_all", [
            ModulePlan("all", "tt", quant_method="none"),
        ]),
    ]

    recipes = []
    for quantized in [False, True]:
        for with_lora in ([False, True] if include_lora else [False]):
            for label, plans in base:
                new_plans = []
                for p in plans:
                    if quantized:
                        qmethod = (
                            "tensorllm_pre_reconstruct_rtn"
                            if p.decomposition_method.startswith("tensorllm")
                            else "rtn_symmetric"
                        )
                        new_plans.append(ModulePlan(
                            p.target,
                            p.decomposition_method,
                            rank=p.rank,
                            quant_method=qmethod,
                            quant_bits=int(quant_bits),
                            params=dict(p.params or {}),
                        ))
                    else:
                        new_plans.append(ModulePlan(
                            p.target,
                            p.decomposition_method,
                            rank=p.rank,
                            quant_method="none",
                            quant_bits=None,
                            params=dict(p.params or {}),
                        ))

                full_label = (
                    label
                    + (f"__rtn{quant_bits}" if quantized else "")
                    + ("__lora" if with_lora else "")
                )
                recipes.append(RecipeSpec(
                    label=full_label,
                    plans=new_plans,
                    with_lora=bool(with_lora),
                    lora_quant_bits=(int(quant_bits) if with_lora and quantized else None),
                    benchmark_with_dense_reconstruction=benchmark_with_dense_reconstruction,
                ))
    return recipes


RECIPES = make_16x_recipes(
    quant_bits=BITWIDTH,
    include_lora=True,
    benchmark_with_dense_reconstruction=BENCHMARK_WITH_DENSE_RECONSTRUCTION,
)

recipes_df = pd.DataFrame([
    {
        'label': r.label,
        'with_lora': r.with_lora,
        'lora_quant_bits': r.lora_quant_bits,
        'benchmark_with_dense_reconstruction': r.benchmark_with_dense_reconstruction,
        'plans': [asdict(p) for p in r.plans],
    }
    for r in RECIPES
])
display(recipes_df)

## New decomposition modes

Laser SVD, per-head TT, per-head Tucker (appended to RECIPES above).

In [ ]:
# ── New decomposition modes ──────────────────────────────────────────────
# These rely on src/head_decomp_methods.py and the patched
# apply_decomposition_plan dispatch in modular_compression_experiment.py.

# Per-head TT order (smaller than full-layer 12 because head_dim is small)
PER_HEAD_TT_ORDER = 4

# Tucker ranks for per-head Tucker
PER_HEAD_TUCKER_HIDDEN_RANK = 64
PER_HEAD_TUCKER_HEAD_DIM_RANK = 16


def make_new_method_recipes(
    *,
    benchmark_with_dense_reconstruction: bool = True,
) -> list:
    """
    Laser:
      laser_mha                – TruncatedSVD on QKVO, reconstructed dense
      laser_ffn                – TruncatedSVD on MLP
      laser_all                – Laser on both MHA and MLP

    Per-head:
      tt_per_head_mha          – TT per attention head (QKVO)
      tucker_per_head_mha      – Tucker per attention head (QKVO)
      tucker_per_head__tt_mlp  – Per-head Tucker MHA + TT MLP

    TensorLLM Tucker (standard 4D QKVO stacking / shape-grouped FFN):
      tensorllm_mha            – Tucker on MHA only
      tensorllm_ffn            – Tucker on FFN only
      tensorllm_mha_ffn        – Tucker on MHA + Tucker on FFN

    TensorLLM Tucker modified (all-but-last factor shared):
      tensorllm_mha_sep_last         – Shared hidden+head-dim factors, per-proj core
      tensorllm_mha_sep_last__ffn    – Modified Tucker MHA + Tucker FFN
    """
    bases = [
        
        # ── Laser ───────────────────────────────────────────────────────────
        ("laser_mha", [
            ModulePlan("mha", "laser", rank=1024, quant_method="none"),
        ]),
        ("laser_ffn", [
            ModulePlan("mlp", "laser", rank=1024, quant_method="none"),
        ]),
        # ── Per-head ────────────────────────────────────────────────────────
        ("tt_per_head_mha", [
            ModulePlan(
                "mha", "tt_per_head", quant_method="none",
                params={"tt_order": PER_HEAD_TT_ORDER},
            ),
        ]),
        ("tucker_per_head_mha", [
            ModulePlan(
                "mha", "tucker_per_head", quant_method="none",
                params={
                    "hidden_rank": PER_HEAD_TUCKER_HIDDEN_RANK,
                    "head_dim_rank": PER_HEAD_TUCKER_HEAD_DIM_RANK,
                },
            ),
        ]),
        # ── TensorLLM Tucker (standard) ─────────────────────────────────────
        ("tensorllm_mha", [
            ModulePlan("mha", "tensorllm_tucker_mha", quant_method="none"),
        ]),
        ("tensorllm_ffn", [
            ModulePlan("mlp", "tensorllm_tucker_mlp", quant_method="none"),
        ]),
        ("tensorllm_mha_ffn", [
            ModulePlan("mha", "tensorllm_tucker_mha", quant_method="none"),
            ModulePlan("mlp", "tensorllm_tucker_mlp", quant_method="none"),
        ]),
        # ── TensorLLM Tucker modified (all-but-last factor shared) ──────────
        ("tensorllm_mha_sep_last", [
            ModulePlan("mha", "tensorllm_tucker_mha_sep_last", quant_method="none"),
        ]),
        ("tensorllm_mha_sep_last__ffn", [
            ModulePlan("mha", "tensorllm_tucker_mha_sep_last", quant_method="none"),
            ModulePlan("mlp", "tensorllm_tucker_mlp", quant_method="none"),
        ]),
        
        ("tt_ffn", [
            ModulePlan("mlp", "tt", quant_bits=None)
        ])
    ]

    bases = [
        ("tt_ffn", [
            ModulePlan("mlp", "tt", quant_bits=None)
        ])
    ]

    recipes = []
    for label, plans in bases:
        recipes.append(RecipeSpec(
            label=label,
            plans=plans,
            with_lora=False,
            benchmark_with_dense_reconstruction=benchmark_with_dense_reconstruction,
        ))
    return recipes


NEW_RECIPES = make_new_method_recipes(
    benchmark_with_dense_reconstruction=BENCHMARK_WITH_DENSE_RECONSTRUCTION,
)

# Append to the global RECIPES list (built by make_16x_recipes above)
RECIPES = NEW_RECIPES

print(f"Total recipes: {len(RECIPES)}  (base: {len(RECIPES)-len(NEW_RECIPES)}, new: {len(NEW_RECIPES)})")
display(pd.DataFrame([
    {
        "label": r.label,
        "plans": [f"{p.target}/{p.decomposition_method}" for p in r.plans],
    }
    for r in NEW_RECIPES
]))


## Sanity check target expansion

In [ ]:
for cfg in RUNNER_CONFIGS:
    model_spec = get_model_architecture(cfg.model_arch_key)
    print('MODEL:', cfg.model_name)
    print('Groups:', model_spec.module_groups)
    target_set = TARGET_SETS_BY_MODEL[cfg.model_name][0]
    specs = make_target_specs(target_set, model_spec)
    display(pd.DataFrame(specs))

## Run experiments

You can interrupt the cell and rerun it. Completed rows are detected from `results` and skipped.

In [ ]:
checkpoint_candidates = []

if OUTPUT_JSON.exists():
    checkpoint_candidates.append(OUTPUT_JSON)

checkpoint_candidates.extend(sorted(OUTPUT_DIR.glob("*__partial_results.json")))

checkpoint_candidates = [p for p in checkpoint_candidates if p.exists()]

if checkpoint_candidates:
    checkpoint_path = max(checkpoint_candidates, key=lambda p: p.stat().st_mtime)
    print("Loading checkpoint:", checkpoint_path)

    results, old_status_df, old_payload = load_results_checkpoint(checkpoint_path)
    loss_histories = {}

    print("Restored result rows:", len(results))
    try:
        display(pd.DataFrame(results)[["model_name", "target_set", "method_label", "rank"]].tail(20))
    except:
        pass

    if len(old_status_df):
        print("Old status counts:")
        display(old_status_df.groupby("status", dropna=False).size().reset_index(name="count"))
else:
    print("No checkpoint found; starting fresh.")
    results = []
    loss_histories = {}
    old_status_df = pd.DataFrame()

In [ ]:
if 'results' not in globals():
    results = []
if 'loss_histories' not in globals():
    loss_histories = {}

all_status = []
pb = tqdm(RUNNER_CONFIGS)
for cfg in pb:
    pb.set_description(f'Model {cfg.model_name}')
    target_sets = TARGET_SETS_BY_MODEL[cfg.model_name]
    results, loss_histories, status_df = run_experiment_grid(
        target_sets=target_sets,
        recipes=RECIPES,
        ranks=RANKS,
        cfgs=[cfg],
        results=results,
        loss_histories=loss_histories,
    )
    all_status.append(status_df)

pb.close()
run_status_df = pd.concat(all_status, ignore_index=True) if all_status else pd.DataFrame()

In [ ]:
run_status_df = pd.concat(all_status, ignore_index=True) if all_status else pd.DataFrame()

results_df = pd.DataFrame(results)

if len(results_df) > 0 and "resume_key" in results_df.columns:
    before = len(results_df)
    results_df["_resume_key_str"] = results_df["resume_key"].apply(
        lambda x: json.dumps(x, sort_keys=True, default=str)
    )
    results_df = (
        results_df
        .drop_duplicates("_resume_key_str", keep="last")
        .drop(columns=["_resume_key_str"])
    )
    after = len(results_df)

    if after < before:
        print(f"Removed {before - after} duplicate rows.")

    results = results_df.to_dict(orient="records")

display(run_status_df)
display(results_df)

if len(run_status_df):
    display(run_status_df.groupby('status', dropna=False).size().reset_index(name='count'))
    failed_df = run_status_df[run_status_df['status'] == 'failed']
    if len(failed_df):
        print('Failed runs:')
        display(failed_df)

saved_json, saved_csv = save_results(
    OUTPUT_JSON,
    results_df=results_df,
    results=results,
    cfgs=RUNNER_CONFIGS,
    target_sets=[ts for cfg in RUNNER_CONFIGS for ts in TARGET_SETS_BY_MODEL[cfg.model_name]],
    recipes=RECIPES,
    status_df=run_status_df,
)

print("Saved JSON:", saved_json)
print("Saved CSV :", saved_csv)

## Results dataframe

In [ ]:
results_df = pd.DataFrame(results)

if len(results_df) > 0 and 'resume_key' in results_df.columns:
    before = len(results_df)
    results_df['_resume_key_str'] = results_df['resume_key'].apply(lambda x: json.dumps(x, sort_keys=True, default=str))
    results_df = results_df.drop_duplicates('_resume_key_str', keep='last').drop(columns=['_resume_key_str'])
    after = len(results_df)
    if after < before:
        print(f'Removed {before - after} duplicate rows.')
        results = results_df.to_dict(orient='records')

display(results_df)

compact_cols = [
    'model_name', 'target_set', 'method_label', 'rank',
    'ppl_wikitext2', 'wikitext2_ppl',
    'affected_modules_compression_ratio',
    'mha_in_affected_blocks_compression_ratio',
    'mlp_in_affected_blocks_compression_ratio',
    'affected_blocks_compression_ratio',
    'overall_no_embeddings_compression_ratio',
    'overall_with_embeddings_compression_ratio',
    'hotpot_test_acc', 'hotpot_test_logloss',
]
display(results_df[[c for c in compact_cols if c in results_df.columns]])

In [ ]:
cos = []
norm = []

for col in results_df.columns:
    if "cos" in col and 'mean' in col:
        cos.append(col)
    
    if "norm" in col:
        norm.append(col)

In [ ]:
results_df[["model_name", 'method_label'] + cos]